In [ ]:
!pip uninstall -y peft accelerate transformers
!pip install \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  scikit-learn \
  sentencepiece


Found existing installation: accelerate 0.25.0
Uninstalling accelerate-0.25.0:
  Successfully uninstalled accelerate-0.25.0
Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
Using cached accelerate-0.25.0-py3-none-any.whl (265 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("anson-huang/mirage-news")
print(dataset)
print(dataset["train"].column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})
['image', 'label', 'text']


In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/BERT_MirageNews"
os.makedirs(output_dir, exist_ok=True)

print("Output dir:", output_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/BERT_MirageNews


In [ ]:
import torch
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "bert-base-uncased"

tokenizer = BertTokenizerFast.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def preprocess(example):
    encoding = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    encoding["labels"] = int(example["label"])
    return encoding


encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        auc = 0.5

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint is not None:
    print("Resuming from:", last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Training from scratch")
    trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Training from scratch


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.379600,0.261281,0.901600,0.935009,0.863200,0.897671,0.957134
2,0.245400,0.276779,0.907200,0.920661,0.891200,0.905691,0.961497
3,0.185000,0.332646,0.906800,0.927670,0.882400,0.904469,0.960022
4,0.101100,0.390474,0.912000,0.927032,0.894400,0.910423,0.959481
5,0.059900,0.441003,0.909200,0.914842,0.902400,0.908578,0.961827


In [ ]:
metrics = trainer.evaluate()
print(metrics)

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Model saved to:", output_dir)


{'eval_loss': 0.39047372341156006, 'eval_accuracy': 0.912, 'eval_precision': 0.9270315091210614, 'eval_recall': 0.8944, 'eval_f1': 0.9104234527687296, 'eval_auc': 0.95948064, 'eval_runtime': 12.4958, 'eval_samples_per_second': 200.068, 'eval_steps_per_second': 6.322, 'epoch': 5.0}
Model saved to: /content/drive/MyDrive/BERT_MirageNews
